<a href="https://colab.research.google.com/github/Vicodwer/Day-37/blob/main/Day_37.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
pip install pandas scikit-learn requests

# TMDB Genre Classification (Text Classification)
Dataset: TMDB Top Rated Movies (Page 3)
Task: Cleanup, Preprocessing, Representation, Modeling, Evaluation

## Dataset Source
API Link:
https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=3

In [11]:
import requests
import pandas as pd
import numpy as np
import re
from collections import Counter

# Fetch data
url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=3"
data = requests.get(url).json()
df = pd.DataFrame(data['results'])

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean_overview'] = df['overview'].apply(clean_text)

# GENRE MAPPING

In [12]:
genre_map = {
    28: "Action", 12: "Adventure", 16: "Animation",
    35: "Comedy", 80: "Crime", 18: "Drama",
    10751: "Family", 14: "Fantasy", 36: "History",
    27: "Horror", 10402: "Music", 9648: "Mystery",
    10749: "Romance", 878: "Sci-Fi",
    53: "Thriller", 10752: "War", 37: "Western"
}

df['genres'] = df['genre_ids'].apply(lambda ids: [genre_map.get(i, "Unknown") for i in ids])

# TF-IDF FROM SCRATCH

In [14]:
docs = df['clean_overview'].tolist()
tokenized = [doc.split() for doc in docs]
vocab = list(set(word for doc in tokenized for word in doc))
vocab_index = {word:i for i, word in enumerate(vocab)}

N = len(docs)

idf = {}
for word in vocab:
    df_count = sum(1 for doc in tokenized if word in doc)
    idf[word] = np.log((N + 1) / (df_count + 1)) + 1


tfidf_matrix = np.zeros((N, len(vocab)))

for i, doc in enumerate(tokenized):
    tf = Counter(doc)
    for word, count in tf.items():
        j = vocab_index[word]
        tfidf_matrix[i, j] = (count / len(doc)) * idf[word]

## COSINE SIMILARITY SEARCH

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=5):
    query = clean_text(query).split()
    q_vec = np.zeros(len(vocab))

    tf = Counter(query)
    for word, count in tf.items():
        if word in vocab_index:
            j = vocab_index[word]
            q_vec[j] = (count / len(query)) * idf[word]

    sims = cosine_similarity([q_vec], tfidf_matrix)[0]
    top_idx = sims.argsort()[-top_k:][::-1]

    return df.iloc[top_idx][['title','overview']]


results = search("wireless earbuds battery life poor")
print(results)

                           title  \
18  Violet Evergarden: The Movie   
15                         Ikiru   
2                      Inception   
6       The Silence of the Lambs   
16                The Wild Robot   

                                             overview  
18  As the world moves on from the war and technol...  
15  Kanji Watanabe is a middle-aged man who has wo...  
2   Cobb, a skilled thief who commits corporate es...  
6   Clarice Starling is a top student at the FBI's...  
16  After a shipwreck, an intelligent robot called...  


# TF-IDF vs SKLEARN

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
sklearn_tfidf = vectorizer.fit_transform(df['clean_overview']).toarray()

# Match dimensions
min_cols = min(sklearn_tfidf.shape[1], tfidf_matrix.shape[1])

diff = tfidf_matrix[:, :min_cols] - sklearn_tfidf[:, :min_cols]
l2_diff = np.linalg.norm(diff)

print("L2 Difference:", l2_diff)

L2 Difference: 4.882661287778911


# MODELING (SVM)

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genres'])

X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, y, test_size=0.2, random_state=42)

model = OneVsRestClassifier(LinearSVC())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         1
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.00      0.00      0.00         3
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         1
          11       0.00      0.00      0.00         0
          12       0.00      0.00      0.00         1

   micro avg       0.00      0.00      0.00         8
   macro avg       0.00      0.00      0.00         8
weighted avg       0.00      0.00      0.00         8
 samples avg       0.00      0.00      0.00         8



/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 8 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to c

Q1(d)

Word with highest TF-IDF (example answer):

The word with the highest average TF-IDF score is typically a rare but highly informative word such as “betrayal” or “rebellion.” It ranks highest because it appears in very few documents (high IDF) but occurs multiple times in specific contexts (high TF), making it highly discriminative.

Q2(a) MANUAL CALCULATION



Let:

TF(fabric, Doc_42) = 3 / 100 = 0.03
Total documents = 10,000
Documents containing "fabric" = 50

IDF:

IDF = log((10000 + 1) / (50 + 1)) + 1
    = log(10001 / 51) + 1
    ≈ log(196.09) + 1
    ≈ 5.28 + 1 = 6.28

TF-IDF:

TF-IDF = 0.03 × 6.28 = 0.1884

Q2(b)

IDF("the") is close to 0 because it appears in almost every document, making it non-informative.
IDF("embroidery") is high because it appears in very few documents, making it highly distinctive.

Q2(c) REBUTTAL

Using only word frequency fails to account for how common a word is across all documents. TF-IDF improves this by reducing the importance of frequent but uninformative words like “the.” Therefore, TF-IDF provides a more meaningful representation for tasks like search and classification.

# (BONUS) BM25

In [18]:
def bm25_score(tf, df, N, doc_len, avg_dl, k1=1.5, b=0.75):
    idf = np.log((N - df + 0.5) / (df + 0.5) + 1)
    return idf * ((tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_len / avg_dl))))

# Git Hub Repository : https://github.com/Vicodwer/Day-37.git